# 03 - Polyvore / Outfit Compatibility V0 - Audit dataset

Objectif : inspecter un dataset Polyvore/outfit, produire un rapport d'audit, et s'arreter avant tout entrainement.

Le module doit rester aligne avec Fashion V1.1 : `product_type_v0`, `canonical_category`, `outfit_role`.


## 1. Monter Google Drive


In [ ]:
from pathlib import Path

from google.colab import drive

DRIVE_MOUNT = Path('/content/drive')
if (DRIVE_MOUNT / 'MyDrive').exists():
    print('Google Drive deja monte.')
else:
    drive.mount(str(DRIVE_MOUNT))

DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive'
print(f'Drive root: {DRIVE_ROOT}')


## 2. Cloner ou mettre a jour le repo


In [ ]:
import os
import shutil
import subprocess
import sys
from datetime import datetime

REPO_URL = 'https://github.com/MilFhey/fit-outfit-advisor.git'
BRANCH = 'main'
REPO_DIR = Path('/content/fit-outfit-advisor-repo')
PROJECT_DIR = REPO_DIR / 'fit-outfit-advisor'

def clone_fresh_repo():
    if REPO_DIR.exists():
        backup_dir = REPO_DIR.with_name(f'{REPO_DIR.name}_backup_{datetime.now().strftime("%Y%m%d_%H%M%S")}')
        print(f'Repo local non reutilisable : sauvegarde vers {backup_dir}')
        shutil.move(str(REPO_DIR), str(backup_dir))
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print('Repo existant : mise a jour.')
    try:
        subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=str(REPO_DIR), check=True)
        subprocess.run(['git', 'checkout', BRANCH], cwd=str(REPO_DIR), check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=str(REPO_DIR), check=True)
    except subprocess.CalledProcessError as exc:
        print(f'Mise a jour impossible par fast-forward ({exc}). Reclonage propre.')
        clone_fresh_repo()
else:
    clone_fresh_repo()

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Dossier projet absent : {PROJECT_DIR}')

sys.path.insert(0, str(PROJECT_DIR))
print(f'Projet pret : {PROJECT_DIR}')


## 3. Installer les dependances


In [ ]:
requirements_path = PROJECT_DIR / 'requirements.txt'
if not requirements_path.exists():
    raise FileNotFoundError(f'Requirements absent : {requirements_path}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'datasets', 'huggingface_hub'], check=True)


## 4. Creer les dossiers temporaires


In [ ]:
RUNTIME_ROOT = Path('/content/fit-outfit-runtime')
POLYVORE_ROOT = RUNTIME_ROOT / 'polyvore'
HF_CACHE_DIR = POLYVORE_ROOT / 'hf_cache'
LOCAL_DATASET_DIR = POLYVORE_ROOT / 'mvasil_polyvore_outfits'
RAW_HF_FILES_DIR = POLYVORE_ROOT / 'raw_hf_files'
DRIVE_DATASET_DIR = DRIVE_ROOT / 'fit-outfit-advisor' / 'datasets' / 'mvasil_polyvore_outfits'
DRIVE_RAW_HF_FILES_DIR = DRIVE_ROOT / 'fit-outfit-advisor' / 'datasets' / 'mvasil_polyvore_outfits_raw_files'
REPORT_DIR = PROJECT_DIR / 'reports'
for directory in [RUNTIME_ROOT, POLYVORE_ROOT, HF_CACHE_DIR, LOCAL_DATASET_DIR, RAW_HF_FILES_DIR, DRIVE_DATASET_DIR, DRIVE_RAW_HF_FILES_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
print(f'Runtime root: {RUNTIME_ROOT}')
print(f'Drive dataset dir: {DRIVE_DATASET_DIR}')
print(f'Drive raw HF files dir: {DRIVE_RAW_HF_FILES_DIR}')


## 5. Configurer la source Hugging Face

Source principale : `mvasil/polyvore-outfits`. Le secret Colab attendu s'appelle `HUGGIN_KEY`.


In [ ]:
HF_DATASET_ID = 'mvasil/polyvore-outfits'
PREFERRED_CONFIGS = ['disjoint', 'nondisjoint']
SAVE_DATASET_TO_DRIVE = True

print(f'Dataset Hugging Face cible : {HF_DATASET_ID}')
print(f'Configs preferees : {PREFERRED_CONFIGS}')


## 6. Charger depuis Drive ou Hugging Face


In [ ]:
import json

from datasets import get_dataset_config_names, load_dataset, load_from_disk
from google.colab import userdata

def is_saved_dataset_dict(path: Path) -> bool:
    return path.exists() and (path / 'dataset_dict.json').exists()


hf_token = userdata.get('HUGGIN_KEY')
drive_saved_configs = sorted(
    path.name for path in DRIVE_DATASET_DIR.iterdir()
    if path.is_dir() and is_saved_dataset_dict(path)
)

if drive_saved_configs:
    source_mode = 'drive_cache'
    available_configs = drive_saved_configs
    selected_configs = [name for name in PREFERRED_CONFIGS if name in drive_saved_configs] or drive_saved_configs
    print('Dataset deja present dans Drive. Aucun telechargement Hugging Face necessaire.')
else:
    source_mode = 'hugging_face_download'
    if not hf_token:
        raise ValueError('Secret Colab HUGGIN_KEY absent et aucune copie Drive exploitable.')
    try:
        available_configs = get_dataset_config_names(HF_DATASET_ID, token=hf_token)
    except Exception as exc:
        print(f'Impossible de lister les configs HF, tentative config par defaut : {exc}')
        available_configs = ['default']
    selected_configs = [name for name in PREFERRED_CONFIGS if name in available_configs]
    if not selected_configs:
        selected_configs = available_configs or ['default']

loaded_datasets = {}
for config_name in selected_configs:
    local_config_dir = LOCAL_DATASET_DIR / config_name
    drive_config_dir = DRIVE_DATASET_DIR / config_name
    if is_saved_dataset_dict(drive_config_dir):
        print(f'Restauration depuis Drive : {drive_config_dir}')
        dataset = load_from_disk(str(drive_config_dir))
        if not is_saved_dataset_dict(local_config_dir):
            dataset.save_to_disk(str(local_config_dir))
            print(f'Copie locale restauree : {local_config_dir}')
    elif is_saved_dataset_dict(local_config_dir):
        print(f'Restauration depuis disque local Colab : {local_config_dir}')
        dataset = load_from_disk(str(local_config_dir))
    else:
        if not hf_token:
            raise ValueError('Secret Colab HUGGIN_KEY absent et config non presente dans Drive.')
        print(f'Telechargement HF : {HF_DATASET_ID} / {config_name}')
        dataset_kwargs = {
            'path': HF_DATASET_ID,
            'token': hf_token,
            'cache_dir': str(HF_CACHE_DIR),
        }
        if config_name != 'default':
            dataset_kwargs['name'] = config_name
        dataset = load_dataset(**dataset_kwargs)
        dataset.save_to_disk(str(local_config_dir))
        if SAVE_DATASET_TO_DRIVE:
            if drive_config_dir.exists():
                shutil.rmtree(drive_config_dir)
            dataset.save_to_disk(str(drive_config_dir))
            print(f'Sauvegarde Drive : {drive_config_dir}')
    loaded_datasets[config_name] = dataset

dataset_root = LOCAL_DATASET_DIR
if not any(dataset_root.iterdir()):
    # Cas restauration depuis Drive sans sauvegarde locale preexistante.
    for config_name, dataset in loaded_datasets.items():
        dataset.save_to_disk(str(dataset_root / config_name))

print(f'Mode source : {source_mode}')
print(f'Dataset root: {dataset_root}')
print(f'Configs disponibles HF : {available_configs}')
print(f'Configs chargees : {list(loaded_datasets)}')
print('Exemples de fichiers sauvegardes:')
for path in list(dataset_root.rglob('*'))[:30]:
    print(path.relative_to(dataset_root))


## 7. Verifier les fichiers HF et le schema charge


In [ ]:
import pandas as pd

from huggingface_hub import hf_hub_download, list_repo_files

repo_files = []
if hf_token:
    try:
        repo_files = list_repo_files(HF_DATASET_ID, repo_type='dataset', token=hf_token)
    except Exception as exc:
        print(f'Impossible de lister les fichiers Hugging Face : {exc}')
else:
    print('Secret HUGGIN_KEY non disponible : verification des fichiers HF ignoree, cache Drive utilise.')

if repo_files:
    repo_files_df = pd.DataFrame({'repo_file': repo_files})
    display(repo_files_df)

def is_raw_metadata_file(repo_file: str) -> bool:
    if repo_file in {'categories.csv', 'polyvore_item_metadata.json', 'polyvore_outfit_titles.json'}:
        return True
    if repo_file.startswith(('disjoint/', 'nondisjoint/', 'maryland_polyvore_hardneg/')):
        return repo_file.endswith(('.json', '.txt', '.p'))
    return False


raw_repo_files = [repo_file for repo_file in repo_files if is_raw_metadata_file(repo_file)]
raw_metadata_files = []
for repo_file in raw_repo_files:
    local_path = RAW_HF_FILES_DIR / repo_file
    drive_path = DRIVE_RAW_HF_FILES_DIR / repo_file
    if drive_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)
        source = 'drive_cache'
    else:
        if not hf_token:
            raise ValueError(f'Secret HUGGIN_KEY absent pour telecharger {repo_file}')
        downloaded_path = Path(hf_hub_download(
            repo_id=HF_DATASET_ID,
            filename=repo_file,
            repo_type='dataset',
            token=hf_token,
            local_dir=str(RAW_HF_FILES_DIR),
        ))
        local_path = downloaded_path
        drive_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_path, drive_path)
        source = 'hugging_face_download'
    raw_metadata_files.append({
        'repo_file': repo_file,
        'local_path': str(local_path),
        'drive_path': str(drive_path),
        'source': source,
        'size_bytes': int(local_path.stat().st_size),
    })

raw_metadata_files_df = pd.DataFrame(raw_metadata_files)
display(raw_metadata_files_df)

def summarize_raw_metadata_file(file_info: dict) -> dict:
    path = Path(file_info['local_path'])
    summary = {
        'repo_file': file_info['repo_file'],
        'source': file_info['source'],
        'size_bytes': file_info['size_bytes'],
        'suffix': path.suffix.lower(),
        'readable': True,
    }
    try:
        if path.suffix.lower() == '.csv':
            sample = pd.read_csv(path, nrows=5000, low_memory=False)
            summary.update({
                'kind': 'csv',
                'sample_shape': list(sample.shape),
                'columns': list(sample.columns),
                'id_like_columns': [col for col in sample.columns if 'id' in str(col).lower()],
                'category_like_columns': [col for col in sample.columns if any(key in str(col).lower() for key in ['cat', 'type', 'label', 'name'])],
                'outfit_like_columns': [col for col in sample.columns if any(key in str(col).lower() for key in ['outfit', 'set'])],
            })
        elif path.suffix.lower() == '.json':
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                sample_records = data[:5]
                sample_keys = sorted({key for item in sample_records if isinstance(item, dict) for key in item.keys()})
                record_count = len(data)
            elif isinstance(data, dict):
                sample_records = list(data.values())[:5]
                sample_keys = list(data.keys())[:20]
                record_count = len(data)
            else:
                sample_records = []
                sample_keys = []
                record_count = None
            nested_keys = sorted({
                key
                for record in sample_records
                if isinstance(record, dict)
                for key in record.keys()
            })
            summary.update({
                'kind': 'json',
                'top_level_type': type(data).__name__,
                'record_count': record_count,
                'sample_keys': sample_keys,
                'nested_sample_keys': nested_keys,
                'id_like_keys': [key for key in sample_keys + nested_keys if 'id' in str(key).lower()],
                'category_like_keys': [key for key in sample_keys + nested_keys if any(token in str(key).lower() for token in ['cat', 'type', 'label', 'name'])],
                'outfit_like_keys': [key for key in sample_keys + nested_keys if any(token in str(key).lower() for token in ['outfit', 'set'])],
            })
        elif path.suffix.lower() == '.txt':
            lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
            summary.update({
                'kind': 'txt',
                'line_count': len(lines),
                'sample_lines': lines[:5],
            })
        elif path.suffix.lower() == '.p':
            summary.update({'kind': 'pickle', 'note': 'Non charge automatiquement dans cette cellule.'})
        else:
            summary.update({'kind': 'other'})
    except Exception as exc:
        summary.update({'readable': False, 'error': repr(exc)})
    return summary

raw_metadata_reports = [summarize_raw_metadata_file(file_info) for file_info in raw_metadata_files]
raw_metadata_reports_df = pd.DataFrame(raw_metadata_reports)
display(raw_metadata_reports_df)

raw_metadata_available = bool(raw_repo_files)
raw_outfit_files_available = any(file_info['repo_file'].endswith(('/train.json', '/valid.json', '/test.json')) for file_info in raw_metadata_files)
raw_item_metadata_available = any(file_info['repo_file'] == 'polyvore_item_metadata.json' for file_info in raw_metadata_files)
raw_categories_available = any(file_info['repo_file'] == 'categories.csv' for file_info in raw_metadata_files)
raw_compatibility_files_available = any('compatibility_' in file_info['repo_file'] for file_info in raw_metadata_files)
raw_metadata_ready_for_schema_audit = raw_outfit_files_available and raw_item_metadata_available and raw_categories_available
print(f'Fichiers raw metadata disponibles : {raw_metadata_available}')
print(f'Pret pour audit schema cooccurrence : {raw_metadata_ready_for_schema_audit}')

dataset_feature_reports = []
for config_name, dataset in loaded_datasets.items():
    for split_name, split_dataset in dataset.items():
        first_row = split_dataset[0] if len(split_dataset) else {}
        dataset_feature_reports.append({
            'config': config_name,
            'split': split_name,
            'row_count': int(len(split_dataset)),
            'features': str(split_dataset.features),
            'first_row_keys': list(first_row.keys()),
            'first_row_value_types': {key: type(value).__name__ for key, value in first_row.items()},
        })

dataset_feature_reports_df = pd.DataFrame(dataset_feature_reports)
display(dataset_feature_reports_df)


## 8. Detecter fichiers et structures candidates


In [ ]:
import pandas as pd

all_files = [path for path in dataset_root.rglob('*') if path.is_file()]
tabular_files = [path for path in all_files if path.suffix.lower() in {'.csv', '.json', '.jsonl'}]
image_files = [path for path in all_files if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]

def classify_file(path: Path) -> str:
    name = path.name.lower()
    if 'outfit' in name or 'set' in name:
        return 'outfit_candidate'
    if 'item' in name or 'product' in name:
        return 'item_candidate'
    if 'category' in name or 'metadata' in name or 'meta' in name:
        return 'metadata_candidate'
    return 'other_tabular'

file_summary = []
for path in tabular_files:
    file_summary.append({
        'relative_path': str(path.relative_to(dataset_root)),
        'suffix': path.suffix.lower(),
        'kind_guess': classify_file(path),
        'size_bytes': path.stat().st_size,
    })

file_summary_df = pd.DataFrame(file_summary).sort_values(['kind_guess', 'relative_path'])
display(file_summary_df.head(80))
print(f'Tabular files: {len(tabular_files)}')
print(f'Image files: {len(image_files)}')


## 9. Inspecter les splits Hugging Face


In [ ]:
hf_split_reports = []
for config_name, dataset in loaded_datasets.items():
    for split_name, split_dataset in dataset.items():
        sample_size = min(5000, len(split_dataset))
        sample_df = split_dataset.select(range(sample_size)).to_pandas() if sample_size else pd.DataFrame()
        columns = list(sample_df.columns)
        hf_split_reports.append({
            'config': config_name,
            'split': split_name,
            'row_count': int(len(split_dataset)),
            'sample_shape': list(sample_df.shape),
            'columns': columns,
            'missing_pct_top': sample_df.isna().mean().sort_values(ascending=False).head(10).round(4).to_dict() if not sample_df.empty else {},
            'id_like_columns': [col for col in columns if 'id' in str(col).lower()],
            'category_like_columns': [col for col in columns if any(key in str(col).lower() for key in ['cat', 'type', 'label', 'name'])],
            'outfit_like_columns': [col for col in columns if any(key in str(col).lower() for key in ['outfit', 'set'])],
        })

hf_split_reports_df = pd.DataFrame(hf_split_reports)
display(hf_split_reports_df[['config', 'split', 'row_count', 'sample_shape', 'id_like_columns', 'category_like_columns', 'outfit_like_columns']])

has_outfit_columns = any(report['outfit_like_columns'] for report in hf_split_reports)
has_category_columns = any(report['category_like_columns'] for report in hf_split_reports)
loader_only_cooccurrence_possible = has_outfit_columns and has_category_columns
cooccurrence_baseline_possible = loader_only_cooccurrence_possible or raw_metadata_ready_for_schema_audit
audit_blockers = []
audit_notes = []
if not has_outfit_columns or not has_category_columns:
    audit_notes.append('Le loader datasets expose surtout item_id/image ; les metadata brutes HF doivent etre inspectees.')
if raw_metadata_ready_for_schema_audit:
    audit_decision = 'raw_metadata_available_requires_schema_mapping'
elif loader_only_cooccurrence_possible:
    audit_decision = 'loader_schema_candidate_requires_mapping_validation'
else:
    if not has_outfit_columns:
        audit_blockers.append('Aucune colonne outfit/set detectee dans les splits Hugging Face.')
    if not has_category_columns:
        audit_blockers.append('Aucune colonne categorie/type/label/name detectee dans les splits Hugging Face.')
    if not raw_metadata_ready_for_schema_audit:
        audit_blockers.append('Fichiers metadata bruts insuffisants pour relier outfit_id, item_id et labels.')
    audit_decision = 'not_exploitable_for_cooccurrence_baseline'

print(f'Baseline via loader datasets uniquement : {loader_only_cooccurrence_possible}')
print(f'Baseline via fichiers raw HF : {raw_metadata_ready_for_schema_audit}')
print(f'Baseline cooccurrence possible : {cooccurrence_baseline_possible}')
print(f'Decision audit : {audit_decision}')
for note in audit_notes:
    print(f'Note: {note}')
for blocker in audit_blockers:
    print(f'- {blocker}')


## 10. Inspecter les fichiers tabulaires lisibles


In [ ]:
def read_table_sample(path: Path):
    try:
        if path.suffix.lower() == '.csv':
            return pd.read_csv(path, nrows=5000, low_memory=False)
        if path.suffix.lower() == '.jsonl':
            return pd.read_json(path, lines=True, nrows=5000)
        if path.suffix.lower() == '.json':
            try:
                data = json.loads(path.read_text(encoding='utf-8'))
            except UnicodeDecodeError:
                data = json.loads(path.read_text(encoding='utf-8-sig'))
            if isinstance(data, list):
                return pd.json_normalize(data[:5000])
            if isinstance(data, dict):
                for value in data.values():
                    if isinstance(value, list):
                        return pd.json_normalize(value[:5000])
                return pd.json_normalize(data)
    except Exception as exc:
        return exc
    return ValueError(f'Format non gere: {path}')

table_reports = []
for path in tabular_files[:80]:
    sample = read_table_sample(path)
    if isinstance(sample, Exception):
        table_reports.append({
            'relative_path': str(path.relative_to(dataset_root)),
            'readable': False,
            'error': repr(sample),
        })
        continue
    columns = list(sample.columns)
    table_reports.append({
        'relative_path': str(path.relative_to(dataset_root)),
        'readable': True,
        'sample_shape': list(sample.shape),
        'columns': columns,
        'missing_pct_top': sample.isna().mean().sort_values(ascending=False).head(10).round(4).to_dict(),
        'id_like_columns': [col for col in columns if 'id' in str(col).lower()],
        'category_like_columns': [col for col in columns if any(key in str(col).lower() for key in ['cat', 'type', 'label', 'name'])],
        'outfit_like_columns': [col for col in columns if any(key in str(col).lower() for key in ['outfit', 'set'])],
    })

table_reports_df = pd.DataFrame(table_reports)
display(table_reports_df[['relative_path', 'readable', 'sample_shape', 'id_like_columns', 'category_like_columns', 'outfit_like_columns']].head(80))


## 11. Inspecter la configuration Outfit V1 brouillon


In [ ]:
from src.mappings.polyvore_mapping import load_outfit_v1_config, validate_outfit_v1_config

OUTFIT_CONFIG_PATH = PROJECT_DIR / 'config' / 'outfit_v1_config.json'
outfit_config = load_outfit_v1_config(OUTFIT_CONFIG_PATH)
validate_outfit_v1_config(outfit_config)
print(json.dumps(outfit_config, indent=2, ensure_ascii=False))
if outfit_config.get('status') == 'draft_requires_dataset_inspection':
    print('Config brouillon : aucun entrainement autorise.')


## 12. Generer le rapport d'audit


In [ ]:
report = {
    'version': 'polyvore_v0_dataset_audit',
    'dataset_root': str(dataset_root),
    'source': {
        'hugging_face_dataset_id': HF_DATASET_ID,
        'hugging_face_secret_name': 'HUGGIN_KEY',
        'available_configs': available_configs,
        'selected_configs': list(loaded_datasets),
        'source_mode': source_mode,
        'drive_cache_used': source_mode == 'drive_cache',
        'drive_dataset_dir': str(DRIVE_DATASET_DIR),
        'repo_files': repo_files,
        'raw_hf_files_dir': str(RAW_HF_FILES_DIR),
        'drive_raw_hf_files_dir': str(DRIVE_RAW_HF_FILES_DIR),
    },
    'file_counts': {
        'total_files': len(all_files),
        'tabular_files': len(tabular_files),
        'image_files': len(image_files),
    },
    'tabular_file_summary': file_summary,
    'table_reports': table_reports,
    'hf_split_reports': hf_split_reports,
    'dataset_feature_reports': dataset_feature_reports,
    'raw_metadata_files': raw_metadata_files,
    'raw_metadata_reports': raw_metadata_reports,
    'raw_metadata_available': raw_metadata_available,
    'raw_outfit_files_available': raw_outfit_files_available,
    'raw_item_metadata_available': raw_item_metadata_available,
    'raw_categories_available': raw_categories_available,
    'raw_compatibility_files_available': raw_compatibility_files_available,
    'raw_metadata_ready_for_schema_audit': raw_metadata_ready_for_schema_audit,
    'outfit_config_status': outfit_config.get('status'),
    'taxonomy_alignment_required': ['product_type_v0', 'canonical_category', 'outfit_role'],
    'loader_only_cooccurrence_possible': loader_only_cooccurrence_possible,
    'cooccurrence_baseline_possible': cooccurrence_baseline_possible,
    'dataset_exploitable_for_outfit_v0': cooccurrence_baseline_possible,
    'audit_decision': audit_decision,
    'audit_blockers': audit_blockers,
    'audit_notes': audit_notes,
    'training_executed': False,
    'next_decision': (
        'Inspecter raw_metadata_reports puis construire mapping Polyvore -> Fashion V1 avant baseline cooccurrence.'
        if cooccurrence_baseline_possible
        else 'Source insuffisante pour baseline cooccurrence: chercher metadata outfits/items avec outfit_id et labels.'
    ),
}

REPORT_PATH = REPORT_DIR / 'polyvore_v0_dataset_audit.json'
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Rapport ecrit : {REPORT_PATH}')

drive_report_dir = DRIVE_ROOT / 'fit-outfit-advisor' / 'reports'
drive_report_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(REPORT_PATH, drive_report_dir / REPORT_PATH.name)
print(f'Rapport copie Drive : {drive_report_dir / REPORT_PATH.name}')


## 13. Audit schema/mapping Polyvore -> Fashion V1

Cette cellule exploite les fichiers raw HF deja telecharges pour construire le rapport de mapping avant toute baseline. Elle ne lance aucun entrainement et ne modifie pas `config/outfit_v1_config.json`.


In [ ]:
import importlib

import src.analysis.analyze_polyvore_v0_schema_mapping as polyvore_schema_mapping
from src.mappings.fashion_v1_mapping import load_fashion_v1_class_config

importlib.invalidate_caches()
polyvore_schema_mapping = importlib.reload(polyvore_schema_mapping)

SCHEMA_MAPPING_REPORT_PATH = REPORT_DIR / 'polyvore_v0_schema_mapping_audit.json'
schema_raw_root = RAW_HF_FILES_DIR if RAW_HF_FILES_DIR.exists() else DRIVE_RAW_HF_FILES_DIR
current_commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(REPO_DIR), text=True).strip()
print(f'Commit repo utilise : {current_commit}')

fashion_config = load_fashion_v1_class_config()
mapping_sanity_checks = {
    'outerwear': 'outerwear',
    'hoodies': 'outerwear',
    'vests': 'outerwear',
    'capri cropped pants': 'trousers',
    'converse chuck taylor all star': 'sports_shoes',
}
sanity_rows = []
for label, expected_product_type in mapping_sanity_checks.items():
    detected = polyvore_schema_mapping.detect_product_type(label, fashion_config)
    sanity_rows.append({
        'label': label,
        'expected_product_type_v0': expected_product_type,
        'detected_status': detected.get('status'),
        'detected_product_type_v0': detected.get('product_type_v0'),
        'reason': detected.get('reason'),
    })
sanity_df = pd.DataFrame(sanity_rows)
display(sanity_df)
bad_sanity_rows = sanity_df[
    (sanity_df['detected_status'] != 'mapped')
    | (sanity_df['detected_product_type_v0'] != sanity_df['expected_product_type_v0'])
]
if not bad_sanity_rows.empty:
    print('Sanity check mapping en echec. Le rapport schema/mapping ne sera pas regenere.')
    print('Action: relance la cellule 2 pour recloner le repo, puis relance cette cellule 13.')
    display(bad_sanity_rows)
else:
    schema_mapping_report = polyvore_schema_mapping.build_schema_mapping_audit(
        raw_root=schema_raw_root,
        dataset_audit_path=REPORT_PATH,
    )

    SCHEMA_MAPPING_REPORT_PATH.write_text(
        json.dumps(schema_mapping_report, indent=2, ensure_ascii=False),
        encoding='utf-8',
    )
    shutil.copy2(SCHEMA_MAPPING_REPORT_PATH, drive_report_dir / SCHEMA_MAPPING_REPORT_PATH.name)

    print(f'Rapport schema/mapping ecrit : {SCHEMA_MAPPING_REPORT_PATH}')
    print(f'Rapport schema/mapping copie Drive : {drive_report_dir / SCHEMA_MAPPING_REPORT_PATH.name}')
    print(f"Decision schema/mapping : {schema_mapping_report['audit_decision']}")
    print(f"Raw root utilise : {schema_mapping_report.get('raw_root')}")
    print(f"Items lies aux metadata : {schema_mapping_report.get('linkage', {}).get('linked_item_count')}")
    print(f"Mappings proposes : {len(schema_mapping_report.get('mapping_proposal', []))}")
    print(f"Labels exclus documentes : {len(schema_mapping_report.get('excluded_labels', []))}")

    mapping_preview = pd.DataFrame(schema_mapping_report.get('mapping_proposal', [])[:30])
    excluded_preview = pd.DataFrame(schema_mapping_report.get('excluded_labels', [])[:30])
    display(mapping_preview)
    display(excluded_preview)


## 14. Arret volontaire avant entrainement


In [ ]:
raise SystemExit(
    'Audit Polyvore termine. Aucun entrainement lance. '
    'Envoie reports/polyvore_v0_dataset_audit.json a Codex pour valider le mapping.'
)
